## Spark Session for Hudi on MinIO

This sets up Spark to use **Apache Hudi** with **MinIO** as the storage backend.

- Adds Hudi support via `hudi-spark3.5-bundle`
- Enables Hudi SQL features
- Connects to MinIO using S3A configs
- Increases memory to avoid Java heap errors
- Reduces shuffle partitions for local use



In [1]:
import os
from pyspark.sql import SparkSession

packages = ",".join([
    "org.apache.hudi:hudi-spark3.5-bundle_2.12:1.0.2",   
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "com.amazonaws:aws-java-sdk-bundle:1.12.530",
])
# Create SparkSession with Hudi + S3A (MinIO) support
spark = (
    SparkSession.builder
      .appName("Hudi Lakehouse on MinIO")
      # Hudi SQL support
      .config("spark.jars.packages", packages)
      .config("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
      .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
       # Mandatory serializer for Hudi
      .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
      # S3A configuration for MinIO access
      .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
      .config("spark.hadoop.fs.s3a.path.style.access", "true")
      .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
      .config("spark.hadoop.fs.s3a.access.key", os.environ["MINIO_ROOT_USER"])
      .config("spark.hadoop.fs.s3a.secret.key", os.environ["MINIO_ROOT_PASSWORD"])
      .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
      # Prevent Java heap errors when writing/upserting
      .config("spark.driver.memory", "4g") 
      .config("spark.executor.memory", "4g")
      # Reduce shuffle overhead for local runs
      .config("spark.sql.shuffle.partitions", "8")
      .getOrCreate()
)
print("Spark:", spark.version)

Spark: 3.5.3


## Create Hudi Table in S3 (MinIO)

This creates a Hudi table `people.staff_hudi` using the **Copy-on-Write (COW)** storage type.

- Partitioned by `partitionpath` (derived from `country`)
- Uses `uuid` as the primary key
- `ts` (derived from `date`) is the pre-combine field (used to deduplicate records)
- Data is stored in MinIO at `s3a://bronze/hudi/people/staff`

In [2]:
spark.sql("CREATE DATABASE IF NOT EXISTS people")

spark.sql("""
CREATE TABLE IF NOT EXISTS people.staff_hudi (
  name STRING,
  country STRING,
  residence STRING,
  role STRING,
  date STRING,
  ts TIMESTAMP,
  uuid STRING,
  partitionpath STRING
)
USING hudi
PARTITIONED BY (partitionpath)
TBLPROPERTIES (
  type = 'cow', -- или 'mor'
  primaryKey = 'uuid',
  preCombineField = 'ts'
)
LOCATION 's3a://bronze/hudi/people/staff'
""")

DataFrame[]

## Insert Data into Hudi Table

This inserts data into the `people.staff_hudi` table from `data.parquet`, adding required fields:

- `uuid`: a unique ID for each row (Hudi primary key)
- `ts`: timestamp for preCombine logic, derived from the `date` column
- `partitionpath`: derived from `country` to use as the partition column

⚠️ In the original NYC taxi version of this notebook, this insert **initially failed** with `java.lang.OutOfMemoryError: Java heap space`.  
To fix it, memory and shuffle settings were adjusted:

```python
.config("spark.driver.memory", "4g") 
.config("spark.executor.memory", "4g")
.config("spark.sql.shuffle.partitions", "8")

In [3]:
spark.sql("""
INSERT INTO people.staff_hudi
SELECT
  *,
  uuid() AS uuid,
  CAST(date AS TIMESTAMP) AS ts,
  country AS partitionpath
FROM parquet.`s3a://raw-data/data.parquet`
""")

DataFrame[]

In [ ]:
spark.sql("""
SELECT * FROM people.staff_hudi ;
""").show(truncate=True)

## Preview Hudi Table with Metadata Columns

This query shows a few rows from the `people.staff_hudi` table, including internal Hudi metadata fields:

- `_hoodie_commit_time`: when the record was written
- `_hoodie_commit_seqno`: commit sequence number
- `_hoodie_record_key`: the primary key (`uuid`)
- `_hoodie_partition_path`: the partition used
- `_hoodie_file_name`: the file storing the row

These metadata fields are automatically managed by Hudi and useful for debugging and tracking data lineage.

In [ ]:
spark.sql("SELECT _hoodie_commit_time, _hoodie_commit_seqno, _hoodie_record_key, _hoodie_partition_path, _hoodie_file_name, name, role FROM people.staff_hudi LIMIT 10").show()

## List Hudi Partitions

Lists all partition values registered for `people.staff_hudi` in the catalog

In [ ]:
spark.sql("SHOW PARTITIONS people.staff_hudi;").show()

## Show Hudi Commit History

This command lists all commits made to the `people.staff_hudi` table.  
Each commit represents an insert, update, or delete operation and includes:

- Commit time
- Operation type (e.g. insert, upsert)
- Total number of records written

In [ ]:
spark.sql("CALL show_commits(table => 'people.staff_hudi')").show(truncate=False)


# Simulate New Batch Ingestion into Hudi Table

Since `data.parquet` is a single file, a new batch is simulated by re-inserting a deterministic
subset of it — every row where `country = 'brazil'` — into the `people.staff_hudi` table.

As before, it adds required fields:

- `ts`: used for preCombine logic (derived from `date`)
- `uuid`: unique ID for the primary key  
- `partitionpath`: determines where the data will be stored (derived from `country`)


In [ ]:
spark.sql("""
INSERT INTO people.staff_hudi
SELECT
  *,
  CAST(date AS TIMESTAMP) AS ts,
  uuid() AS uuid,
  country AS partitionpath
FROM parquet.`s3a://raw-data/data.parquet` WHERE country = 'brazil';
""")

## Verify New Commit After Insert

This command shows the updated commit history of the `people.staff_hudi` table.

A new commit should appear, reflecting the recent insert of the `brazil` subset.  
Each entry includes the commit timestamp, operation type, and number of records written.

In [ ]:
spark.sql("CALL show_commits(table => 'people.staff_hudi')").show(truncate=False)


## Time Travel in Hudi

To read the Hudi table as it was at a specific point in time, set the `as.of.instant` option to a past commit time.

Replace commit_time with a valid value from show_commits. This allows you to view the table state as of that commit.


In [ ]:
commit_time = "20250810152622768"   # replace with a real commit_time from show_commits
df = spark.read.format("hudi") \
    .option("as.of.instant", commit_time) \
    .load("s3a://bronze/hudi/people/staff/")
df.show(5)

## Add and Populate New Column in Hudi Table

A new column `role_group` is added to the `people.staff_hudi` table to categorize staff by role.

Steps:
1. `ADD COLUMNS` adds the new column.
2. `REFRESH TABLE` ensures Spark sees the updated schema.
3. `UPDATE` fills `role_group` with values:
   - `'leadership'` for `tech lead`  
   - `'analytics'` for `data analyst`  
   - `'engineering'` for everything else

Finally, the table schema and value distribution are checked.

In [ ]:
spark.sql("ALTER TABLE people.staff_hudi ADD COLUMNS (role_group STRING)")

spark.sql("REFRESH TABLE people.staff_hudi")

spark.sql("""
UPDATE people.staff_hudi
SET role_group = CASE
  WHEN role = 'tech lead' THEN 'leadership'
  WHEN role = 'data analyst' THEN 'analytics'
  ELSE 'engineering'
END
WHERE role IS NOT NULL
""")

spark.sql("DESCRIBE TABLE people.staff_hudi").show(truncate=False)
spark.sql("SELECT role_group, COUNT(*) FROM people.staff_hudi GROUP BY role_group").show()


### Attempt to Drop a Column from a Hudi Table

I tried to remove the `role_group` column from the Hudi table using:

```python
spark.sql("ALTER TABLE people.staff_hudi DROP COLUMNS (role_group);")
```

However, this failed with:

AnalysisException: [UNSUPPORTED_FEATURE.TABLE_OPERATION] 
The feature is not supported: Table `spark_catalog`.`people`.`staff_hudi` does not support DROP COLUMN. 
Please check the current catalog and namespace to make sure the qualified table name is expected, 
and also check the catalog implementation which is configured by "spark.sql.catalog".

## Adding a New Column to a Hudi Table on the Fly (Writer-Side Schema Evolution)

Hudi supports **schema evolution** when writing data — meaning you can add new columns directly in your writer DataFrame, and Hudi will update the table schema automatically.  

The following PySpark example adds a new nullable numeric column `name_length` (the length of each person's `name`), generates a UUID record key, ensures Hudi-required fields (`recordkey`, `precombine`, and `partitionpath`), and also includes a `role_group` column set to `NULL`

Important:

This approach only works during write (schema evolution in writer).
If you want to add a column using SQL, you must use ALTER TABLE ... ADD COLUMNS (...).

In [ ]:
import uuid
from pyspark.sql.functions import col, lit, udf, length as slen
from pyspark.sql.types import StringType

# UDF to generate UUID strings
@udf(returnType=StringType())
def gen_uuid():
    return str(uuid.uuid4())

# Load original data
df = spark.read.parquet("s3a://raw-data/data.parquet")

# Add a NEW nullable column: name_length, plus Hudi-required fields
df_with_new = (
    df.withColumn("name_length", slen(col("name")))
      .withColumn("ts", col("date").cast("timestamp"))
      .withColumn("uuid", gen_uuid())
      .withColumn("partitionpath", col("country"))
)

# Match table schema
df_fixed = df_with_new.withColumn("role_group", lit(None).cast("string"))

# Write with schema evolution on (adds name_length automatically)
(df_fixed.write.format("hudi")
  .option("hoodie.datasource.write.operation", "upsert")
  .option("hoodie.datasource.write.recordkey.field", "uuid")
  .option("hoodie.datasource.write.precombine.field", "ts")
  .option("hoodie.datasource.write.partitionpath.field", "partitionpath")
  .option("hoodie.schema.on.write.enable", "true")
  .option("hoodie.datasource.write.reconcile.schema", "true")
  .option("hoodie.write.set.null.for.missing.columns", "true")
  .option("hoodie.datasource.write.hive_style_partitioning", "true")
  .mode("append")
  .save("s3a://bronze/hudi/people/staff"))

## Inspecting a Hudi Table via Snapshot Read

To explore the current schema and sample data in a Hudi table, you can perform a **snapshot read** by loading the table path directly into a DataFrame.  
Once loaded, you register it as a temporary view and then use standard SQL commands to:

1. **Describe the table** — view all columns, their data types, and nullability.
2. **Select specific columns** — retrieve `role_group`, `name_length`, and `partitionpath` to confirm that recently added fields are present and populated (or `NULL` for older records).

In [ ]:
df_path = spark.read.format("hudi").load("s3a://bronze/hudi/people/staff")
df_path.createOrReplaceTempView("hudi_staff_snapshot")

spark.sql("DESCRIBE TABLE hudi_staff_snapshot").show(truncate=False)
spark.sql("SELECT role_group, name_length, partitionpath FROM hudi_staff_snapshot LIMIT 10").show()

## Partition Evolution in Hudi

Apache Hudi **does not support partition evolution**.

Once a table is created with a specific partition column (e.g. `partitionpath`), it cannot be changed or updated later.  
Changing the partitioning requires creating a new table with the desired partition spec and rewriting the data.
